# <font color= #003366> **Análisis de Sentimiento** </font>
- <Strong> Subject: </Strong>  <font color="blue">`Deep Learning` </font>
- <Strong> Final Project: </Strong>  <font color="blue">`BERT` </font>

<div style="display: flex; align-items: center;">
    <div style="flex: 1;">
        <img src="https://oci02.img.iteso.mx/Identidades-De-Instancia/ITESO/Logos%20ITESO/Logo-ITESO-Principal.jpg" width="300">
    </div>
</div>

___

## <font color=  #003366> **Introducción**</font>

Aqui va texto

---

In [20]:
from datasets import load_dataset
from collections import Counter

In [4]:
dataset = load_dataset("amazon_polarity")

df_train = dataset['train'].shuffle(seed=42).select(range(10000))
df_test = dataset['test'].shuffle(seed=42).select(range(2000))

df_train = df_train.rename_column("label", "labels")
df_test = df_test.rename_column("label", "labels")

split = df_train.train_test_split(test_size=0.2, seed=42)

df_train = split["train"]
df_val = split["test"]

def combinar(example):
    return {
        "text": example["title"] + " " + example["content"]
    }

df_train = df_train.map(combinar)
df_test = df_test.map(combinar)
df_val = df_val.map(combinar)

df_train = df_train.remove_columns(["title", "content"])
df_test = df_test.remove_columns(["title", "content"])
df_val = df_val.remove_columns(["title", "content"])

c:\Users\sarah\anaconda3\envs\escuela\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sarah\.cache\huggingface\hub\datasets--amazon_polarity. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Map: 100%|██████████| 2000/2000 [00:00<00:00, 9366.10 examples/s]


In [7]:
df_train.head()

,labels,text
0,1,great music The music was great and I absolute...
1,1,Love it! I get a lot of compliments on this ha...
2,0,Very Little That's New Not a bad read if you'r...
3,1,A great tool My husband and I have used our bl...
4,0,Don't Bother! If you're looking to fish in the...


In [17]:
df_train.isnull().sum()

labels       0
text         0
num_words    0
dtype: int64

In [18]:
df_train.duplicated().sum()

np.int64(0)

There is no duplicates or null values which means we can continue with our exploration 

In [8]:
df_train["labels"].value_counts()

labels
1    4005
0    3995
Name: count, dtype: int64

We can observe that 3995 reviews are **negative** and 4005 are **positive**


In [9]:
df_train["text"].str.len().describe()

count    8000.000000
mean      429.370125
std       237.047158
min       100.000000
25%       229.000000
50%       381.000000
75%       595.000000
max      1014.000000
Name: text, dtype: float64

The shortest review has 100 characters and the longest has 1014, probable if a lot of the shortest have a hundred, it could be because of a minimun character requirement.

In [13]:
df_train["num_words"] = df_train["text"].str.split().str.len()

df_train["num_words"].describe()

count    8000.000000
mean       78.224250
std        42.513979
min        15.000000
25%        42.000000
50%        70.000000
75%       108.000000
max       212.000000
Name: num_words, dtype: float64

We can appreciate that the average number of words in the reviews is around 78, with a maximum of 212. This suggests that the reviews are relatively short, which may have implications for the choice of model and preprocessing steps.

In [15]:
df_train.sort_values("num_words", ascending=False).head()

,labels,text,num_words
1761,0,HUGE DISAPPOINTMENT I ORDERED 6 OF THEM FOR CH...,212
7792,1,2 Of the Greatest Guitar Players Of ALL TIME F...,204
6738,0,AliceSmack First off I would like to say that ...,199
2293,0,Delonghi DCM485 Thermal Coffee Maker First let...,199
5747,1,"A solid workout! I own 30 day shred, Banish Fa...",194


Here are **5** of the largest reviews, we can see that the lenght doesn't affect in any mean the sentiment

In [16]:
df_train.sort_values("num_words", ascending=True).head()

,labels,text,num_words
7163,1,Love it! Great quality video.Great sound.Small...,15
7575,1,Review of book shipment/receipt Book arrived a...,15
4005,1,Spiritual Journey Babette's Feast is a glowing...,15
7814,1,Excellent book An excellent resource for the o...,15
4044,1,Technology and how it works Well written expla...,16


And here are 5 of the shortest, although every one of them is positive this doesn't mean that because a review is short it'd be positive

In [21]:
all_words = " ".join(df_train["text"]).lower().split()

Counter(all_words).most_common(10)

[('the', 31701),
 ('and', 16922),
 ('a', 15751),
 ('i', 15663),
 ('to', 15114),
 ('of', 12490),
 ('this', 11159),
 ('is', 11038),
 ('it', 10392),
 ('in', 7313)]

Here are the 10 most common words in the dataset. Most of them are stopwords such as “the”, “and”, “a”, and “to”, which is expected in natural language text dataset